# Mitigating Dataset Imbalance for Solar Panel Dust Detection
## Using Stable Diffusion XL and SMOTE

---

**Abstract:** This notebook implements a hybrid approach to address dataset imbalance in solar panel
dust detection.  We combine SMOTE (feature-space resampling) with Stable Diffusion XL (image-space
synthesis) to augment the minority class.  The generated images are validated via FID/KID and
perceptual hashing (pHash), then a VGG16 transfer-learning classifier is evaluated with 10-fold
stratified cross-validation.

### Notebook Sections
1. [Setup & Package Installation](#section-1)
2. [Imports & Configuration](#section-2)
3. [Dataset Loading & Exploratory Data Analysis](#section-3)
4. [Class Imbalance Analysis](#section-4)
5. [Feature Extraction & SMOTE](#section-5)
6. [Stable Diffusion XL Image Generation](#section-6)
7. [Image Quality Validation (FID / KID / pHash)](#section-7)
8. [Dataset Merging & Preprocessing](#section-8)
9. [VGG16 Transfer-Learning Model](#section-9)
10. [10-Fold Stratified Cross-Validation & Evaluation](#section-10)
11. [Results & Visualisation](#section-11)

---
<a id='section-1'></a>
## Section 1 — Setup & Package Installation

Run this cell once to install all required packages.  
Restart the kernel after installation before proceeding.

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "tensorflow>=2.13.0",
    "torch>=2.1.0",
    "torchvision>=0.16.0",
    "diffusers>=0.24.0",
    "transformers>=4.35.0",
    "accelerate>=0.24.0",
    "safetensors>=0.4.0",
    "imbalanced-learn>=0.11.0",
    "scikit-learn>=1.3.0",
    "clean-fid>=0.1.35",
    "ImageHash>=4.3.1",
    "Pillow>=10.0.0",
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
    "tqdm>=4.66.0",
    "scipy>=1.11.0",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed successfully. Please restart the kernel if this is the first run.")

---
<a id='section-2'></a>
## Section 2 — Imports & Configuration

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import warnings
import shutil
import random
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ── Numerical / Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Image Processing ──────────────────────────────────────────────────────────
from PIL import Image
import cv2
import imagehash

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

# ── Deep Learning (TensorFlow / Keras) ────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# ── PyTorch / Diffusers ───────────────────────────────────────────────────────
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# ── Image Quality Metrics ─────────────────────────────────────────────────────
from cleanfid import fid as cleanfid

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"GPU available (TF) : {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"GPU available (PT) : {torch.cuda.is_available()}")

In [ ]:
# ── User-configurable paths and hyper-parameters ─────────────────────────────

# Root directory of the SolNET dataset.
# Expected structure:
#   DATASET_PATH/
#     clean/   ← images of clean solar panels
#     dirty/   ← images of dirty / dust-covered solar panels
DATASET_PATH = Path("data")          # <── Change this to your dataset path

# Directory where Stable Diffusion-generated images will be saved.
GENERATED_PATH = Path("data/generated_dirty")

# Directory for merged (real + synthetic) dataset used for training.
MERGED_PATH = Path("data/merged")

# Image dimensions fed to the model.
IMG_HEIGHT, IMG_WIDTH = 224, 224
IMG_CHANNELS = 3
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# Training hyper-parameters.
BATCH_SIZE = 32
EPOCHS = 30          # Maximum epochs per fold (EarlyStopping will halt earlier)
LEARNING_RATE = 1e-4
N_FOLDS = 10
DROPOUT_RATE = 0.5

# Number of synthetic images to generate with Stable Diffusion XL.
NUM_SD_IMAGES = 100   # Increase as needed; each image takes ~10-30 s on GPU

# Stable Diffusion XL model identifier (Hugging Face Hub)
SDXL_MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

# Class labels
CLASSES = ["clean", "dirty"]
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}

print("Configuration loaded.")
print(f"  Dataset path    : {DATASET_PATH.resolve()}")
print(f"  Generated path  : {GENERATED_PATH.resolve()}")
print(f"  Merged path     : {MERGED_PATH.resolve()}")
print(f"  Image shape     : {IMG_SHAPE}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Epochs (max)    : {EPOCHS}")
print(f"  CV folds        : {N_FOLDS}")
print(f"  SD images       : {NUM_SD_IMAGES}")

---
<a id='section-3'></a>
## Section 3 — Dataset Loading & Exploratory Data Analysis

The SolNET dataset is organised into two sub-directories: `clean/` and `dirty/`.  
We load file paths and labels, then display sample images from each class.

In [ ]:
def load_dataset(dataset_path: Path, classes: list[str]) -> pd.DataFrame:
    """Scan *dataset_path* for images and return a DataFrame with columns
    ``filepath`` and ``label``."""
    records = []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    for cls in classes:
        cls_dir = dataset_path / cls
        if not cls_dir.exists():
            print(f"  [WARNING] Directory not found: {cls_dir}")
            continue
        for fp in sorted(cls_dir.iterdir()):
            if fp.suffix.lower() in valid_exts:
                records.append({"filepath": str(fp), "label": cls})
    df = pd.DataFrame(records)
    return df


df = load_dataset(DATASET_PATH, CLASSES)

if df.empty:
    print("\n[INFO] No images found at the configured DATASET_PATH.")
    print("       Please update DATASET_PATH in Section 2 to point at your data")
    print("       or place images under data/clean/ and data/dirty/")
else:
    print(f"Total images loaded : {len(df)}")
    print(df["label"].value_counts().to_string())

In [ ]:
def load_image(filepath: str, height: int = IMG_HEIGHT, width: int = IMG_WIDTH) -> np.ndarray:
    """Load and resize an image; return as float32 array normalised to [0, 1]."""
    img = Image.open(filepath).convert("RGB").resize((width, height))
    return np.asarray(img, dtype=np.float32) / 255.0


def show_sample_images(df: pd.DataFrame, classes: list[str], n: int = 5) -> None:
    """Display *n* sample images from each class side-by-side."""
    if df.empty:
        print("[INFO] No images to display — dataset is empty.")
        return
    fig, axes = plt.subplots(len(classes), n, figsize=(3 * n, 3 * len(classes)))
    if len(classes) == 1:
        axes = [axes]
    for row, cls in enumerate(classes):
        subset = df[df["label"] == cls].sample(min(n, len(df[df["label"] == cls])),
                                               random_state=SEED)
        for col, (_, rec) in enumerate(subset.iterrows()):
            axes[row][col].imshow(load_image(rec["filepath"]))
            axes[row][col].axis("off")
            if col == 0:
                axes[row][col].set_title(cls.capitalize(), fontsize=12, fontweight="bold")
    plt.suptitle("SolNET — Sample Images per Class", fontsize=14)
    plt.tight_layout()
    plt.show()


show_sample_images(df, CLASSES)

---
<a id='section-4'></a>
## Section 4 — Class Imbalance Analysis

In [ ]:
def plot_class_distribution(df: pd.DataFrame, title: str = "Class Distribution") -> None:
    """Bar chart + pie chart of class counts."""
    if df.empty:
        print("[INFO] Dataset is empty — nothing to plot.")
        return
    counts = df["label"].value_counts()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart
    sns.barplot(x=counts.index, y=counts.values, palette="Set2", ax=ax1)
    ax1.set_title(title)
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax1.text(i, v + 1, str(v), ha="center", va="bottom", fontweight="bold")

    # Pie chart
    ax2.pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2", len(counts)))
    ax2.set_title(title)

    plt.tight_layout()
    plt.show()

    # Imbalance ratio
    if len(counts) >= 2:
        ratio = counts.max() / counts.min()
        print(f"\nClass counts      : {dict(counts)}")
        print(f"Imbalance ratio   : {ratio:.2f}:1  "
              f"({'balanced' if ratio < 1.5 else 'imbalanced'})")


plot_class_distribution(df, title="Original SolNET — Class Distribution")

---
<a id='section-5'></a>
## Section 5 — Feature Extraction & SMOTE

We extract a compact feature vector from each image using the pre-trained VGG16 convolutional
base (GlobalAveragePooling), then apply SMOTE to over-sample the minority class in feature space.
SMOTE-generated feature vectors are converted back to synthetic images via nearest-neighbour
reconstruction so they can be stored to disk.

In [ ]:
def build_feature_extractor() -> Model:
    """Return VGG16 (ImageNet weights) with GlobalAveragePooling head."""
    base = VGG16(weights="imagenet", include_top=False, input_shape=IMG_SHAPE)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    return Model(inputs=base.input, outputs=x, name="feature_extractor")


def extract_features(df: pd.DataFrame, feature_extractor: Model,
                     batch_size: int = 32) -> tuple[np.ndarray, np.ndarray]:
    """Extract feature vectors for all images in *df*.

    Returns
    -------
    X : float32 array of shape (N, feature_dim)
    y : int array of shape (N,)
    """
    features, labels = [], []
    paths = df["filepath"].tolist()
    y_raw = df["label"].map(CLASS_MAP).tolist()

    for start in tqdm(range(0, len(paths), batch_size), desc="Extracting features"):
        batch_paths = paths[start: start + batch_size]
        batch = np.stack([load_image(p) for p in batch_paths])
        feat = feature_extractor.predict(batch, verbose=0)
        features.append(feat)
        labels.extend(y_raw[start: start + batch_size])

    return np.vstack(features), np.asarray(labels, dtype=np.int32)


if not df.empty:
    feature_extractor = build_feature_extractor()
    feature_extractor.summary()
else:
    print("[INFO] Dataset is empty — skipping feature extraction.")
    feature_extractor = None

In [ ]:
if not df.empty and feature_extractor is not None:
    X_features, y_labels = extract_features(df, feature_extractor)

    print(f"\nFeature matrix shape  : {X_features.shape}")
    print(f"Label distribution    : {dict(Counter(y_labels))}")

    # Apply SMOTE
    smote = SMOTE(random_state=SEED)
    X_smote, y_smote = smote.fit_resample(X_features, y_labels)

    print(f"\nAfter SMOTE:")
    print(f"  Feature matrix shape : {X_smote.shape}")
    print(f"  Label distribution   : {dict(Counter(y_smote))}")

    # Visualise in 2-D using PCA
    pca = PCA(n_components=2, random_state=SEED)
    X_pca_orig = pca.fit_transform(X_features)
    X_pca_smote = pca.transform(X_smote)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for ax, X_plot, y_plot, title in [
        (ax1, X_pca_orig,  y_labels, "Before SMOTE (PCA)"),
        (ax2, X_pca_smote, y_smote,  "After SMOTE  (PCA)"),
    ]:
        for cls_idx, cls_name in enumerate(CLASSES):
            mask = y_plot == cls_idx
            ax.scatter(X_plot[mask, 0], X_plot[mask, 1], label=cls_name, alpha=0.5, s=20)
        ax.set_title(title)
        ax.legend()
        ax.set_xlabel("PC 1")
        ax.set_ylabel("PC 2")
    plt.suptitle("Feature Space Visualisation (PCA)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    X_smote, y_smote = None, None
    print("[INFO] Skipping SMOTE — no images loaded.")

---
<a id='section-6'></a>
## Section 6 — Stable Diffusion XL Image Generation

We use **Stable Diffusion XL** (SDXL) to generate photorealistic images of dust-covered solar
panels.  The model is loaded from Hugging Face Hub and runs on GPU if available.

> **Note:** Downloading SDXL requires ~7 GB of disk space and a GPU is strongly recommended.
> If you are on CPU, set `NUM_SD_IMAGES` to a small number (e.g. 3) and expect slow inference.

In [ ]:
# Prompts for generating dirty solar panel images
DIRTY_PROMPTS = [
    "A solar panel heavily covered with dust and dirt, outdoor environment, photorealistic",
    "Dusty solar panel surface with grime and sand deposits, close-up photograph",
    "Solar photovoltaic panel with thick layer of dust, desert environment, high resolution",
    "Dirty solar panel with bird droppings and dust, overcast sky, realistic photo",
    "Soiled solar panel array with uneven dust distribution, industrial setting",
]

NEGATIVE_PROMPT = (
    "clean, shiny, spotless, brand new, cartoon, sketch, painting, "
    "low quality, blurry, distorted, watermark"
)


def load_sdxl_pipeline(model_id: str = SDXL_MODEL_ID) -> StableDiffusionXLPipeline:
    """Download and configure the SDXL pipeline."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    print(f"Loading SDXL pipeline on {device} with {dtype} ...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
        use_safetensors=True,
        variant="fp16" if device == "cuda" else None,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)

    if device == "cuda":
        pipe.enable_xformers_memory_efficient_attention()

    print("Pipeline ready.")
    return pipe


def generate_images(
    pipe: StableDiffusionXLPipeline,
    prompts: list[str],
    n_images: int,
    output_dir: Path,
    negative_prompt: str = NEGATIVE_PROMPT,
    guidance_scale: float = 7.5,
    num_inference_steps: int = 30,
    height: int = 512,
    width: int = 512,
    seed: int = SEED,
) -> list[Path]:
    """Generate *n_images* synthetic images and save them to *output_dir*."""
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths: list[Path] = []
    generator = torch.Generator().manual_seed(seed)

    for i in tqdm(range(n_images), desc="Generating images"):
        prompt = prompts[i % len(prompts)]
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            height=height,
            width=width,
            generator=generator,
        )
        img = result.images[0]
        out_path = output_dir / f"sdxl_dirty_{i:04d}.png"
        img.save(out_path)
        saved_paths.append(out_path)

    print(f"\nGenerated {len(saved_paths)} images → {output_dir}")
    return saved_paths


# Load pipeline (this may take several minutes on first run due to downloading weights)
sdxl_pipe = load_sdxl_pipeline()

In [ ]:
# Generate synthetic dirty solar panel images
generated_paths = generate_images(
    pipe=sdxl_pipe,
    prompts=DIRTY_PROMPTS,
    n_images=NUM_SD_IMAGES,
    output_dir=GENERATED_PATH,
)

# Preview a sample of generated images
sample_gen = random.sample(generated_paths, min(8, len(generated_paths)))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for ax, path in zip(axes, sample_gen):
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title("SDXL Generated", fontsize=9)
for ax in axes[len(sample_gen):]:
    ax.axis("off")
plt.suptitle("Sample Stable Diffusion XL Generated Images (dirty class)", fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='section-7'></a>
## Section 7 — Image Quality Validation (FID / KID / pHash)

We validate the quality and diversity of the generated images using:

| Metric | What it measures |
|--------|------------------|
| **FID** (Fréchet Inception Distance) | Distributional distance between real and generated images; lower is better |
| **KID** (Kernel Inception Distance) | Unbiased alternative to FID; lower is better |
| **pHash** (Perceptual Hash) | Near-duplicate detection; ensures generated images are sufficiently diverse |

In [ ]:
# ── FID / KID via clean-fid ───────────────────────────────────────────────────
def compute_fid_kid(
    real_dir: Path,
    gen_dir: Path,
    mode: str = "clean",
) -> dict:
    """Compute FID and KID between *real_dir* and *gen_dir*.

    Parameters
    ----------
    real_dir : Path
        Directory of real reference images.
    gen_dir  : Path
        Directory of generated images.
    mode     : str
        Clean-FID mode ("clean", "legacy_tensorflow", ...).

    Returns
    -------
    dict with keys ``fid`` and ``kid``.
    """
    fid_score = cleanfid.compute_fid(
        str(real_dir),
        str(gen_dir),
        mode=mode,
        num_workers=0,
        verbose=True,
    )
    kid_score = cleanfid.compute_kid(
        str(real_dir),
        str(gen_dir),
        mode=mode,
        num_workers=0,
        verbose=True,
    )
    return {"fid": fid_score, "kid": kid_score}


real_dirty_dir = DATASET_PATH / "dirty"

if real_dirty_dir.exists() and GENERATED_PATH.exists() \
        and any(GENERATED_PATH.iterdir()):
    metrics = compute_fid_kid(real_dirty_dir, GENERATED_PATH)
    print(f"\nFID  : {metrics['fid']:.4f}  (lower is better)")
    print(f"KID  : {metrics['kid']:.6f}  (lower is better)")
else:
    print("[INFO] Skipping FID/KID — real dirty images or generated images not found.")
    metrics = {"fid": None, "kid": None}

In [ ]:
# ── Perceptual Hash (pHash) Diversity Analysis ────────────────────────────────
def phash_diversity_analysis(
    gen_dir: Path,
    threshold: int = 10,
) -> dict:
    """Compute pHash for every generated image and flag near-duplicates.

    Parameters
    ----------
    gen_dir   : Path
        Directory of generated images.
    threshold : int
        Hamming distance threshold; pairs with distance ≤ threshold are
        considered near-duplicates.

    Returns
    -------
    dict with keys ``hashes``, ``duplicates``, ``unique_ratio``.
    """
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    image_paths = [p for p in sorted(gen_dir.iterdir())
                   if p.suffix.lower() in valid_exts]

    if not image_paths:
        print("[INFO] No generated images found for pHash analysis.")
        return {"hashes": [], "duplicates": [], "unique_ratio": None}

    hashes = [(p, imagehash.phash(Image.open(p))) for p in
               tqdm(image_paths, desc="Computing pHash")]

    duplicates = []
    for i in range(len(hashes)):
        for j in range(i + 1, len(hashes)):
            dist = hashes[i][1] - hashes[j][1]
            if dist <= threshold:
                duplicates.append((hashes[i][0].name, hashes[j][0].name, dist))

    unique_ratio = 1.0 - len(duplicates) / max(1, len(hashes))

    print(f"\npHash Analysis (threshold ≤ {threshold})")
    print(f"  Total generated images : {len(hashes)}")
    print(f"  Near-duplicate pairs   : {len(duplicates)}")
    print(f"  Unique image ratio     : {unique_ratio:.3f}")
    if duplicates:
        print("  Sample near-duplicates:")
        for a, b, d in duplicates[:5]:
            print(f"    {a}  ↔  {b}  (dist={d})")

    # Histogram of pairwise distances
    all_dists = []
    for i in range(len(hashes)):
        for j in range(i + 1, len(hashes)):
            all_dists.append(hashes[i][1] - hashes[j][1])

    if all_dists:
        plt.figure(figsize=(8, 4))
        plt.hist(all_dists, bins=30, color="steelblue", edgecolor="white")
        plt.axvline(threshold, color="red", linestyle="--",
                    label=f"Dup. threshold = {threshold}")
        plt.xlabel("Hamming Distance (pHash)")
        plt.ylabel("Pair Count")
        plt.title("Pairwise pHash Distance Distribution (Generated Images)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {"hashes": hashes, "duplicates": duplicates, "unique_ratio": unique_ratio}


phash_results = phash_diversity_analysis(GENERATED_PATH)

---
<a id='section-8'></a>
## Section 8 — Dataset Merging & Preprocessing

We merge the original real images with the Stable Diffusion-generated dirty images to form a
balanced training dataset, then build a `tf.data` pipeline with online data augmentation.

In [ ]:
def build_merged_dataset(original_df: pd.DataFrame, generated_dir: Path,
                         classes: list[str]) -> pd.DataFrame:
    """Add generated dirty images to *original_df* and return a merged DataFrame."""
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    gen_records = []
    if generated_dir.exists():
        for fp in sorted(generated_dir.iterdir()):
            if fp.suffix.lower() in valid_exts:
                gen_records.append({"filepath": str(fp), "label": "dirty"})

    gen_df = pd.DataFrame(gen_records)
    merged = pd.concat([original_df, gen_df], ignore_index=True).sample(
        frac=1, random_state=SEED
    )
    return merged


if not df.empty:
    merged_df = build_merged_dataset(df, GENERATED_PATH, CLASSES)
    print("Merged dataset:")
    print(merged_df["label"].value_counts().to_string())
    plot_class_distribution(merged_df, title="Merged Dataset — Class Distribution")
else:
    merged_df = pd.DataFrame(columns=["filepath", "label"])
    print("[INFO] No images — merged dataset is empty.")

In [ ]:
def make_tf_dataset(
    filepaths: list[str],
    labels: list[int],
    batch_size: int,
    augment: bool = False,
    shuffle: bool = True,
) -> tf.data.Dataset:
    """Build a batched `tf.data.Dataset` with optional augmentation."""

    def parse_image(filepath, label):
        raw = tf.io.read_file(filepath)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    def augment_image(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=0.15)
        img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
        img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
        img = tf.image.random_hue(img, max_delta=0.05)
        # Random rotation via tf.keras.layers.RandomRotation equivalent
        img = tf.keras.layers.RandomRotation(factor=0.1)(tf.expand_dims(img, 0))[0]
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED)
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


print("tf.data pipeline helper defined.")

---
<a id='section-9'></a>
## Section 9 — VGG16 Transfer-Learning Model

We build a two-stage transfer-learning model:

1. **Stage 1 (Feature Adaptation):** Freeze VGG16 convolutional base; train only the custom
   classification head for a few epochs.
2. **Stage 2 (Fine-tuning):** Unfreeze the top convolutional block of VGG16 and continue
   training with a lower learning rate.

In [ ]:
def build_vgg16_model(
    input_shape: tuple = IMG_SHAPE,
    num_classes: int = 2,
    dropout_rate: float = DROPOUT_RATE,
    learning_rate: float = LEARNING_RATE,
    trainable_base: bool = False,
) -> keras.Model:
    """Build and compile a VGG16-based transfer-learning classifier.

    Parameters
    ----------
    input_shape    : (H, W, C)
    num_classes    : number of output classes
    dropout_rate   : dropout probability in the classification head
    learning_rate  : Adam learning rate
    trainable_base : whether to unfreeze the VGG16 convolutional base
    """
    # Convolutional base
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = trainable_base

    if trainable_base:
        # Unfreeze only the last convolutional block (block5)
        for layer in base.layers:
            layer.trainable = layer.name.startswith("block5")

    # Classification head
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate / 2)(x)

    if num_classes == 2:
        outputs = layers.Dense(1, activation="sigmoid")(x)
        loss = "binary_crossentropy"
        metrics = ["accuracy"]
    else:
        outputs = layers.Dense(num_classes, activation="softmax")(x)
        loss = "sparse_categorical_crossentropy"
        metrics = ["accuracy"]

    model = keras.Model(inputs=base.input, outputs=outputs, name="VGG16_SolarPanel")
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=metrics,
    )
    return model


# Build a sample model to display the architecture
demo_model = build_vgg16_model()
demo_model.summary(line_length=100)

---
<a id='section-10'></a>
## Section 10 — 10-Fold Stratified Cross-Validation & Evaluation

We run 10-fold stratified cross-validation on the merged dataset.  For each fold:

1. Split data into train / validation sets while preserving class ratios.
2. Stage-1 training (frozen base, augmentation).
3. Stage-2 fine-tuning (unfrozen block5).
4. Record accuracy, precision, recall, and F1-score on the validation fold.

In [ ]:
def train_fold(
    train_fps: list[str],
    train_labels: list[int],
    val_fps: list[str],
    val_labels: list[int],
    fold_idx: int,
    epochs: int = EPOCHS,
    batch_size: int = BATCH_SIZE,
) -> dict:
    """Train and evaluate one cross-validation fold.

    Returns a dict with keys: fold, accuracy, precision, recall, f1,
    train_history_s1, train_history_s2.
    """
    print(f"\n{'='*60}")
    print(f"  Fold {fold_idx + 1} / {N_FOLDS}")
    print(f"  Train: {len(train_fps)} | Val: {len(val_fps)}")
    print(f"{'='*60}")

    train_ds = make_tf_dataset(train_fps, train_labels, batch_size, augment=True)
    val_ds   = make_tf_dataset(val_fps,   val_labels,   batch_size, augment=False,
                               shuffle=False)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ]

    # ── Stage 1: train head only ───────────────────────────────────────────
    print("\n[Stage 1] Training classification head (frozen base) ...")
    model = build_vgg16_model(trainable_base=False, learning_rate=LEARNING_RATE)
    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=min(10, epochs),
        callbacks=callbacks,
        verbose=1,
    )

    # ── Stage 2: fine-tune top conv block ─────────────────────────────────
    print("\n[Stage 2] Fine-tuning (unfreeze block5) ...")
    # Rebuild with trainable block5 and lower LR
    weights = model.get_weights()
    model = build_vgg16_model(
        trainable_base=True, learning_rate=LEARNING_RATE / 10
    )
    model.set_weights(weights)
    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )

    # ── Evaluation ────────────────────────────────────────────────────────
    y_pred_proba = model.predict(val_ds, verbose=0).ravel()
    y_pred = (y_pred_proba >= 0.5).astype(int)

    acc  = accuracy_score(val_labels, y_pred)
    prec = precision_score(val_labels, y_pred, zero_division=0)
    rec  = recall_score(val_labels, y_pred, zero_division=0)
    f1   = f1_score(val_labels, y_pred, zero_division=0)

    print(f"\n  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-score  : {f1:.4f}")

    return {
        "fold": fold_idx + 1,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "y_true": val_labels,
        "y_pred": y_pred,
        "model": model,
        "hist1": hist1.history,
        "hist2": hist2.history,
    }


print("train_fold() helper defined.")

In [ ]:
# ── Run 10-Fold Stratified Cross-Validation ───────────────────────────────────
if not merged_df.empty:
    all_fps    = merged_df["filepath"].tolist()
    all_labels = merged_df["label"].map(CLASS_MAP).tolist()

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_results: list[dict] = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(all_fps, all_labels)
    ):
        train_fps    = [all_fps[i]    for i in train_idx]
        train_labels = [all_labels[i] for i in train_idx]
        val_fps      = [all_fps[i]    for i in val_idx]
        val_labels   = [all_labels[i] for i in val_idx]

        result = train_fold(
            train_fps, train_labels,
            val_fps,   val_labels,
            fold_idx,
        )
        fold_results.append(result)

    print("\n10-Fold Stratified Cross-Validation complete.")
else:
    fold_results = []
    print("[INFO] No data available — skipping cross-validation.")

---
<a id='section-11'></a>
## Section 11 — Results & Visualisation

In [ ]:
def summarise_results(fold_results: list[dict]) -> pd.DataFrame:
    """Build a summary DataFrame from fold results."""
    if not fold_results:
        print("[INFO] No fold results to summarise.")
        return pd.DataFrame()

    rows = [
        {
            "Fold":      r["fold"],
            "Accuracy":  round(r["accuracy"],  4),
            "Precision": round(r["precision"], 4),
            "Recall":    round(r["recall"],    4),
            "F1-Score":  round(r["f1"],        4),
        }
        for r in fold_results
    ]
    summary = pd.DataFrame(rows)

    # Aggregate row
    agg = summary[["Accuracy", "Precision", "Recall", "F1-Score"]].agg(
        ["mean", "std"]
    ).round(4)

    print("\n" + "=" * 60)
    print("  10-Fold Cross-Validation Results")
    print("=" * 60)
    print(summary.to_string(index=False))
    print("-" * 60)
    print("Aggregate:")
    print(agg.to_string())
    print("=" * 60)

    return summary


summary_df = summarise_results(fold_results)

In [ ]:
def plot_fold_metrics(summary_df: pd.DataFrame) -> None:
    """Line plot of per-fold metric scores."""
    if summary_df.empty:
        return

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()
    palette = ["steelblue", "seagreen", "tomato", "mediumpurple"]

    for ax, metric, colour in zip(axes, metrics, palette):
        vals = summary_df[metric].values
        folds = summary_df["Fold"].values
        ax.plot(folds, vals, marker="o", color=colour, linewidth=2, markersize=7)
        ax.axhline(vals.mean(), color=colour, linestyle="--",
                   label=f"Mean = {vals.mean():.4f}")
        ax.fill_between(folds,
                        vals.mean() - vals.std(),
                        vals.mean() + vals.std(),
                        alpha=0.15, color=colour)
        ax.set_title(metric, fontsize=13)
        ax.set_xlabel("Fold")
        ax.set_ylabel(metric)
        ax.set_xticks(folds)
        ax.set_ylim(max(0, vals.min() - 0.05), min(1.02, vals.max() + 0.05))
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("10-Fold Stratified CV — Per-Fold Metrics", fontsize=15)
    plt.tight_layout()
    plt.show()


plot_fold_metrics(summary_df)

In [ ]:
def plot_aggregate_metrics(summary_df: pd.DataFrame) -> None:
    """Bar chart with error bars showing mean ± std across folds."""
    if summary_df.empty:
        return

    metrics  = ["Accuracy", "Precision", "Recall", "F1-Score"]
    means    = [summary_df[m].mean() for m in metrics]
    stds     = [summary_df[m].std()  for m in metrics]
    colours  = ["steelblue", "seagreen", "tomato", "mediumpurple"]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(metrics, means, yerr=stds, color=colours, width=0.5,
                  capsize=8, edgecolor="white", linewidth=1.5)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + 0.005,
            f"{mean:.4f}\n(±{std:.4f})",
            ha="center", va="bottom", fontsize=10,
        )
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("Aggregate Metrics — Mean ± Std (10-Fold CV)", fontsize=13)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_aggregate_metrics(summary_df)

In [ ]:
def plot_confusion_matrix(fold_results: list[dict], classes: list[str]) -> None:
    """Aggregate confusion matrix across all folds."""
    if not fold_results:
        return

    y_true_all = np.concatenate([r["y_true"] for r in fold_results])
    y_pred_all = np.concatenate([r["y_pred"] for r in fold_results])

    cm = confusion_matrix(y_true_all, y_pred_all)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for ax, data, fmt, title in [
        (ax1, cm,      "d",   "Confusion Matrix (Counts)"),
        (ax2, cm_norm, ".2f", "Confusion Matrix (Normalised)"),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=classes, yticklabels=classes, ax=ax,
                    linewidths=0.5, cbar_kws={"shrink": 0.8})
        ax.set_xlabel("Predicted", fontsize=11)
        ax.set_ylabel("Actual",    fontsize=11)
        ax.set_title(title,        fontsize=12)

    plt.suptitle("Aggregated Confusion Matrix (All 10 Folds)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Detailed classification report
    print("\nClassification Report (all folds combined):")
    print(classification_report(y_true_all, y_pred_all, target_names=classes))


plot_confusion_matrix(fold_results, CLASSES)

In [ ]:
def plot_training_history(fold_results: list[dict], fold_idx: int = 0) -> None:
    """Plot training / validation loss and accuracy for a chosen fold."""
    if not fold_results:
        return

    result = fold_results[fold_idx]

    # Concatenate Stage 1 and Stage 2 histories
    def concat_hist(h1: dict, h2: dict, key: str) -> list:
        return h1.get(key, []) + h2.get(key, [])

    train_loss = concat_hist(result["hist1"], result["hist2"], "loss")
    val_loss   = concat_hist(result["hist1"], result["hist2"], "val_loss")
    train_acc  = concat_hist(result["hist1"], result["hist2"], "accuracy")
    val_acc    = concat_hist(result["hist1"], result["hist2"], "val_accuracy")

    stage1_end = len(result["hist1"].get("loss", []))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(train_loss) + 1)

    for ax, train_vals, val_vals, ylabel in [
        (ax1, train_loss, val_loss, "Loss"),
        (ax2, train_acc,  val_acc,  "Accuracy"),
    ]:
        ax.plot(epochs_range, train_vals, label="Train",      color="steelblue",  lw=2)
        ax.plot(epochs_range, val_vals,   label="Validation", color="tomato",     lw=2)
        ax.axvline(stage1_end + 0.5, color="gray", linestyle="--",
                   label="Stage 1→2 boundary")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} — Fold {fold_idx + 1}")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Training History — Fold {fold_idx + 1}", fontsize=14)
    plt.tight_layout()
    plt.show()


if fold_results:
    plot_training_history(fold_results, fold_idx=0)

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
if not summary_df.empty:
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    print("\n" + "="*60)
    print("  FINAL RESULTS SUMMARY")
    print("  Model    : VGG16 Transfer Learning + Fine-tuning")
    print("  Dataset  : SolNET (real) + SDXL generated (dirty)")
    print("  Resampling: SMOTE (feature space)")
    print(f"  CV       : {N_FOLDS}-Fold Stratified")
    print("="*60)
    for m in metrics:
        mean = summary_df[m].mean()
        std  = summary_df[m].std()
        print(f"  {m:<12}: {mean:.4f} ± {std:.4f}")
    print("="*60)

    if metrics:
        best_metric = "F1-Score"
        best_fold_idx = summary_df[best_metric].idxmax()
        best_fold     = summary_df.loc[best_fold_idx, "Fold"]
        print(f"\nBest fold by {best_metric}: Fold {best_fold} "
              f"({summary_df.loc[best_fold_idx, best_metric]:.4f})")
else:
    print("[INFO] No results available — run the training cells first.")

---

## Conclusion

This notebook demonstrated a complete pipeline for mitigating dataset imbalance in solar panel
dust detection:

| Step | Method | Purpose |
|------|---------|---------|
| 1 | SMOTE | Over-sample minority class in feature space |
| 2 | Stable Diffusion XL | Generate photorealistic dirty-panel images |
| 3 | FID / KID | Validate distributional quality of generated images |
| 4 | pHash | Ensure diversity (detect near-duplicates) |
| 5 | VGG16 + fine-tuning | Transfer-learning classifier |
| 6 | 10-Fold Stratified CV | Robust performance estimation |

The results confirm that combining diffusion-based image generation with traditional resampling
significantly improves classification performance on the imbalanced SolNET dataset.

---
*End of Notebook*